In [ ]:
# Cell 1: Install dependencies

!pip install -q fastapi uvicorn transformers sentencepiece nest-asyncio requests

In [ ]:
# Cell 2: Model configuration

import os
import torch

MODEL_PATH = "/kaggle/input/datasets/rajdeeppa53/pii-roberta-fine-tuned"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# These should match the settings used during training.
MAX_LENGTH = 256
STRIDE = 48

# Number of encoded windows passed to the model in one forward pass.
INFERENCE_BATCH_SIZE = 16

# Number of input texts tokenized together at a time.
TOKENIZE_BATCH_SIZE = 64

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import secrets
print(secrets.token_hex(32))

In [ ]:
%%writefile /kaggle/working/app.py
# Cell 3: Create app.py

import os
import threading
from contextlib import nullcontext
from typing import List

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification


# --------------------------------------------------
# Configuration
# --------------------------------------------------

MODEL_PATH = os.environ.get(
    "MODEL_PATH",
    "/kaggle/input/your-model-dataset/xlm-roberta-pii-detector"
)

MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "256"))
STRIDE = int(os.environ.get("STRIDE", "48"))
INFERENCE_BATCH_SIZE = int(
    os.environ.get("INFERENCE_BATCH_SIZE", "16")
)
TOKENIZE_BATCH_SIZE = int(
    os.environ.get("TOKENIZE_BATCH_SIZE", "64")
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.set_num_threads(4)


# --------------------------------------------------
# Load model once, when the application starts
# --------------------------------------------------

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=True
)

if not tokenizer.is_fast:
    raise RuntimeError("A fast tokenizer is required for offset mapping.")

print("Loading model...")

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH
)

model.to(DEVICE)
model.eval()

ID2LABEL = {
    int(key): value
    for key, value in model.config.id2label.items()
}

print("Model loaded on:", DEVICE)


# Prevent simultaneous requests from competing for GPU memory.
inference_lock = threading.Lock()


# --------------------------------------------------
# FastAPI
# --------------------------------------------------

app = FastAPI(
    title="Batch PII Detection API",
    version="2.0"
)


class BatchRequest(BaseModel):
    texts: List[str] = Field(min_length=1, max_length=128)


# --------------------------------------------------
# Convert BIO predictions into entities
# --------------------------------------------------

def build_entities(tokens):

    entities = []
    current = None

    def flush():
        nonlocal current

        if current is not None:
            entities.append(current)
            current = None

    for token in tokens:

        label = token["label"]
        start = token["start"]
        end = token["end"]
        score = token["score"]

        if label == "O" or "-" not in label:
            flush()
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix == "B":
            flush()

            current = {
                "type": entity_type,
                "start": start,
                "end": end,
                "score": score
            }

        elif prefix == "I":

            if current is not None and current["type"] == entity_type:
                current["end"] = end
                current["score"] = min(current["score"], score)

            else:
                # Recover gracefully if the model predicts I without B.
                flush()

                current = {
                    "type": entity_type,
                    "start": start,
                    "end": end,
                    "score": score
                }

        else:
            flush()

    flush()

    return entities


# --------------------------------------------------
# Redact entities from original text
# --------------------------------------------------

def redact_text(text, entities):

    result = text

    # Replace from right to left so earlier offsets remain valid.
    for entity in sorted(
        entities,
        key=lambda item: item["start"],
        reverse=True
    ):

        start = entity["start"]
        end = entity["end"]
        entity_type = entity["type"]

        result = (
            result[:start]
            + f"[{entity_type}]"
            + result[end:]
        )

    return result


# --------------------------------------------------
# Run inference for a batch of texts
# --------------------------------------------------

def detect_batch(texts):

    all_results = [[] for _ in texts]

    # Process input texts in smaller groups to control RAM usage.
    for text_group_start in range(0, len(texts), TOKENIZE_BATCH_SIZE):

        text_group_end = min(
            text_group_start + TOKENIZE_BATCH_SIZE,
            len(texts)
        )

        text_group = texts[text_group_start:text_group_end]

        # Create overlapping windows for long texts.
        encoded = tokenizer(
            text_group,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            truncation=True,
            padding=True,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )

        sample_mapping = encoded.pop("overflow_to_sample_mapping")
        offsets = encoded.pop("offset_mapping")

        # Save original character offsets before moving tensors to GPU.
        offsets = offsets.tolist()
        sample_mapping = sample_mapping.tolist()

        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]

        # Each text can produce multiple windows.
        # Keep the best prediction for each unique character span.
        best_predictions = [
            {} for _ in text_group
        ]

        number_of_windows = input_ids.shape[0]

        for window_start in range(
            0,
            number_of_windows,
            INFERENCE_BATCH_SIZE
        ):

            window_end = min(
                window_start + INFERENCE_BATCH_SIZE,
                number_of_windows
            )

            batch_ids = input_ids[window_start:window_end].to(DEVICE)
            batch_mask = attention_mask[window_start:window_end].to(DEVICE)

            with torch.inference_mode():

                if DEVICE.type == "cuda":
                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):
                        outputs = model(
                            input_ids=batch_ids,
                            attention_mask=batch_mask
                        )
                else:
                    outputs = model(
                        input_ids=batch_ids,
                        attention_mask=batch_mask
                    )

                probabilities = torch.softmax(
                    outputs.logits.float(),
                    dim=-1
                )

                scores, predicted_ids = probabilities.max(dim=-1)

            predicted_ids = predicted_ids.cpu().tolist()
            scores = scores.cpu().tolist()

            # Map every window back to its original text.
            for local_window_index in range(
                window_start,
                window_end
            ):

                sample_index = sample_mapping[local_window_index]

                local_index = local_window_index - window_start

                token_ids = predicted_ids[local_index]
                token_scores = scores[local_index]
                token_offsets = offsets[local_window_index]

                for token_index, (start, end) in enumerate(token_offsets):

                    # Ignore special tokens and padding.
                    if start == end:
                        continue

                    if attention_mask[local_window_index, token_index].item() == 0:
                        continue

                    label_id = token_ids[token_index]
                    label = ID2LABEL.get(label_id, "O")
                    score = float(token_scores[token_index])

                    key = (start, end)

                    previous = best_predictions[sample_index].get(key)

                    # Overlapping windows may predict the same token.
                    # Retain the prediction with the higher confidence.
                    if previous is None or score > previous["score"]:

                        best_predictions[sample_index][key] = {
                            "label": label,
                            "start": start,
                            "end": end,
                            "score": score
                        }

        # Convert token predictions to entities for each text.
        for local_index, text in enumerate(text_group):

            tokens = list(best_predictions[local_index].values())

            tokens.sort(
                key=lambda item: (item["start"], item["end"])
            )

            entities = build_entities(tokens)

            # Extract the exact original text for every detected entity.
            for entity in entities:
                entity["text"] = text[
                    entity["start"]:entity["end"]
                ]

            all_results[text_group_start + local_index] = {
                "entities": entities,
                "redacted_text": redact_text(text, entities)
            }

    return all_results


# --------------------------------------------------
# Health endpoint
# --------------------------------------------------

@app.get("/health")
def health():

    return {
        "status": "healthy",
        "device": str(DEVICE),
        "model_loaded": True
    }


# --------------------------------------------------
# Batch PII detection endpoint
# --------------------------------------------------

@app.post("/detect-pii/batch")
def detect_pii_batch(request: BatchRequest):

    texts = request.texts

    if not texts:
        raise HTTPException(
            status_code=400,
            detail="At least one text is required."
        )

    if any(not isinstance(text, str) for text in texts):
        raise HTTPException(
            status_code=422,
            detail="Every item must be a string."
        )

    # Serialize inference requests to avoid GPU contention.
    with inference_lock:
        results = detect_batch(texts)

    return {
        "count": len(texts),
        "results": results
    }


# --------------------------------------------------
# Single-text endpoint (optional)
# --------------------------------------------------

@app.post("/detect-pii")
def detect_pii_single(request: BatchRequest):

    if len(request.texts) != 1:
        raise HTTPException(
            status_code=400,
            detail="Send exactly one text to this endpoint."
        )

    with inference_lock:
        result = detect_batch(request.texts)

    return result[0]

In [ ]:
# Cell 4: Set environment variables

import os

os.environ["MODEL_PATH"] = MODEL_PATH
os.environ["MAX_LENGTH"] = str(MAX_LENGTH)
os.environ["STRIDE"] = str(STRIDE)
os.environ["INFERENCE_BATCH_SIZE"] = str(INFERENCE_BATCH_SIZE)
os.environ["TOKENIZE_BATCH_SIZE"] = str(TOKENIZE_BATCH_SIZE)

In [ ]:
# Cell 5: Start FastAPI

import subprocess
import time
import requests

server_log = open("/kaggle/working/uvicorn.log", "w")

server = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--workers", "1"
    ],
    cwd="/kaggle/working",
    stdout=server_log,
    stderr=subprocess.STDOUT
)

# Wait for the API to become ready.
for attempt in range(60):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=3
        )

        if response.status_code == 200:
            print(response.json())
            break

    except requests.RequestException:
        time.sleep(2)

else:
    print("API failed to start. Check /kaggle/working/uvicorn.log")

In [ ]:
# Cell 6: Install cloudflared

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

In [ ]:
# Cell 7: Start the temporary tunnel

import subprocess
import time
import re
import os

tunnel_log_path = "/kaggle/working/cloudflared.log"

tunnel_log = open(tunnel_log_path, "w")

tunnel = subprocess.Popen(
    [
        "/kaggle/working/cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None

for attempt in range(60):

    time.sleep(2)

    with open(tunnel_log_path, "r") as f:
        logs = f.read()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        public_url = match.group(0)
        break

if public_url:
    print("Your API URL:", public_url)
else:
    print("Tunnel URL not found. Check cloudflared.log")

In [ ]:
# Cell 8: Send multiple texts in one request

texts = [
    "UI\n\nName Age / Gender Contact No. Address Pin code\n\n: Mr. DEVANSH SURANA\n\n: 20 Year(s)/ Male : +916353063347 : 803 RIVERA TOWERS BULIDING 4 LALJI NA… : 395009\n\n",
    "Contact Alice at alice@example.com or 9876543210.",
    "The patient visited the hospital on 12/05/2025.",
    "Venanzius Höttermann Revés was married in July 1995."
]

response = requests.post(
    f"{public_url}/detect-pii/batch",
    json={"texts": texts},
    timeout=120
)

response.raise_for_status()

data = response.json()

for index, result in enumerate(data["results"]):

    print(f"\nText {index + 1}")
    print("Entities:", result["entities"])
    print("Redacted:", result["redacted_text"])